# yolo12n training notebook (selected patients only)

This notebook trains/testing yolo12 nano using only these patients: `p4, p5, p7, p12, p15, p16, p17`.

Pipeline:
1. Collect image-label pairs from selected patient folders
2. Apply the same augmentations used in the reference scripts
3. Split into train/val/test
4. Train + validate + test yolo12n

In [ ]:
# Install required packages for this notebook
%pip install -q ultralytics albumentations opencv-python pandas matplotlib tqdm

## Required folder structure
Expected under `pipeline_medsam2_3d`:
- `p4_extracted_pngs`, `p5_extracted_pngs`, `p7_extracted_pngs`, `p12_extracted_pngs`, `p15_extracted_pngs`, `p16_extracted_pngs`, `p17_extracted_pngs`
- `p4_labels`, `p5_labels`, `p7_labels`, `p12_labels`, `p15_labels`, `p16_labels`, `p17_labels`
- each patient folder has `axial/`, `coronal/`, `sagittal/`
- model weights file: `models/yolo12n/yolo12n.pt`

In [ ]:
import os
import sys
import random
import shutil
from pathlib import Path

import cv2
import numpy as np
from tqdm import tqdm
import albumentations as A
from ultralytics import YOLO

print('Python executable:', sys.executable)

## Optional (Colab): mount Google Drive
Use this if running in Colab and your dataset/project lives in Drive.

In [ ]:
# Colab-safe drive mount
PROJECT_ROOT_OVERRIDE = None

try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore

    drive.mount('/content/drive')

    # Update this path to your Drive location of knee_ct_scan_segmentation
    PROJECT_ROOT_OVERRIDE = Path('/content/drive/MyDrive/knee_ct_scan_segmentation')
    print('Colab detected. PROJECT_ROOT_OVERRIDE =', PROJECT_ROOT_OVERRIDE)
except Exception:
    print('Not running in Colab. Using local paths.')

## Optional (Colab): extract training_data.zip from Drive
Use this if your training folders are packed as `training_data.zip` in Drive.

In [ ]:
import zipfile

# Zip is expected inside pipeline_medsam2_3d in Drive/local project root
BASE_ROOT = PROJECT_ROOT_OVERRIDE if PROJECT_ROOT_OVERRIDE is not None else Path('/home/akshaj/Documents/projects/knee_ct_scan_segmentation')
ZIP_PATH = BASE_ROOT / 'pipeline_medsam2_3d' / 'training_data.zip'

# Since zip contains only patient folders, extract directly into pipeline_medsam2_3d
TARGET_ROOT = BASE_ROOT / 'pipeline_medsam2_3d'

if ZIP_PATH.exists():
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(TARGET_ROOT)
    print('Extracted zip:', ZIP_PATH)
    print('Extracted into:', TARGET_ROOT)
else:
    print('Zip not found:', ZIP_PATH)
    print('Expected location: <project_root>/pipeline_medsam2_3d/training_data.zip')

## Config
Set paths, selected patients, and training hyperparameters.

In [ ]:
# Base project paths
DEFAULT_LOCAL_PROJECT_ROOT = Path('/home/akshaj/Documents/projects/knee_ct_scan_segmentation')
PROJECT_ROOT = PROJECT_ROOT_OVERRIDE if PROJECT_ROOT_OVERRIDE is not None else DEFAULT_LOCAL_PROJECT_ROOT
PIPELINE_DIR = PROJECT_ROOT / 'pipeline_medsam2_3d'

# Only these patients are used
SELECTED_PATIENTS = ['p4', 'p5', 'p7', 'p12', 'p15', 'p16', 'p17']
ALL_VIEWS = ['axial', 'coronal', 'sagittal']

# 4 models: per-view + all views
MODEL_SPECS = {
    'axial': ['axial'],
    'coronal': ['coronal'],
    'sagittal': ['sagittal'],
    'all_views': ['axial', 'coronal', 'sagittal'],
}

# Weights must be inside pipeline_medsam2_3d
WEIGHTS_DIR = PIPELINE_DIR / 'models' / 'yolo12n'
WEIGHTS = WEIGHTS_DIR / 'yolo12n.pt'

# Output roots
DATA_ROOT = PIPELINE_DIR / 'yolo12_selected_patients_dataset'
RUNS_ROOT = PIPELINE_DIR / 'runs_yolo12'

TRAIN_RATIO = 0.85
VAL_RATIO = 0.075
TEST_RATIO = 0.075
SEED = 42

# Same augmentations as reference yolo_aug.py
N_SSR = 3
SHIFT_LIMIT = 0.1
SCALE_LIMIT = 0.15
ROTATE_LIMIT = 30
MIN_VIS = 0.30
MIN_WH = 0.01

# Reuse existing augmented files if present
# True: skip rebuilding aug folders when they already contain files
# False: clear and regenerate augmentations each run
REUSE_AUGMENTATIONS = True

# Training config
IMGSZ = 512
EPOCHS = 100
BATCH = 8
WORKERS = 4

random.seed(SEED)
np.random.seed(SEED)

WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PIPELINE_DIR:', PIPELINE_DIR)
print('WEIGHTS path:', WEIGHTS)
print('Patients:', SELECTED_PATIENTS)
print('Model specs:', MODEL_SPECS)
print('REUSE_AUGMENTATIONS:', REUSE_AUGMENTATIONS)

## Auto-download YOLO12n weights
If `models/yolo12n/yolo12n.pt` is missing, this cell fetches it into the same local/Drive folder used by this notebook.

In [ ]:
def ensure_yolo12n_weights(weights_path: Path):
    """Use Ultralytics-native weight resolution/download and store in target path."""
    weights_path.parent.mkdir(parents=True, exist_ok=True)

    if weights_path.exists() and weights_path.stat().st_size > 0:
        print('Weights found:', weights_path)
        return weights_path

    print('Weights missing. Resolving with Ultralytics...')

    # Ultralytics built-in model resolution/download
    tmp_model = YOLO('yolo12n.pt')

    candidates = [Path('yolo12n.pt')]

    for obj in [tmp_model, getattr(tmp_model, 'model', None)]:
        if obj is None:
            continue
        for attr in ['ckpt_path', 'pt_path']:
            v = getattr(obj, attr, None)
            if v:
                candidates.append(Path(v))

    resolved = None
    for p in candidates:
        if p.exists() and p.is_file() and p.stat().st_size > 0:
            resolved = p
            break

    if resolved is None:
        raise FileNotFoundError('Ultralytics could not resolve yolo12n.pt')

    if resolved.resolve() != weights_path.resolve():
        shutil.copy2(resolved, weights_path)

    print('Weights ready at:', weights_path)
    return weights_path


WEIGHTS = ensure_yolo12n_weights(WEIGHTS)
print('Using weights:', WEIGHTS)

## 1) Collect data from selected patients
Copies only image-label pairs from the selected patient PNG/label folders.

In [ ]:
def reset_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)
    for f in path.glob('*'):
        if f.is_file():
            f.unlink()


def collect_pairs_for_views(target_views, raw_im_dir: Path, raw_lb_dir: Path):
    """Copy matched image-label pairs for selected patients and views."""
    reset_dir(raw_im_dir)
    reset_dir(raw_lb_dir)

    copied = 0
    skipped = 0

    for p in SELECTED_PATIENTS:
        img_root = PIPELINE_DIR / f'{p}_extracted_pngs'
        lb_root = PIPELINE_DIR / f'{p}_labels'

        if not img_root.exists() or not lb_root.exists():
            print(f'[WARN] Missing folder for {p}')
            continue

        for view in target_views:
            view_im = img_root / view
            view_lb = lb_root / view
            if not view_im.exists() or not view_lb.exists():
                print(f'[WARN] Missing view folder: {p}/{view}')
                continue

            image_files = sorted(
                [x for x in view_im.glob('*.png')]
                + [x for x in view_im.glob('*.jpg')]
                + [x for x in view_im.glob('*.jpeg')]
            )

            for im_path in image_files:
                stem = im_path.stem
                lb_path = view_lb / f'{stem}.txt'
                if not lb_path.exists():
                    skipped += 1
                    continue

                out_base = f'{p}_{view}_{stem}'
                shutil.copy2(im_path, raw_im_dir / f'{out_base}{im_path.suffix.lower()}')
                shutil.copy2(lb_path, raw_lb_dir / f'{out_base}.txt')
                copied += 1

    return copied, skipped

## 2) Augmentation (same as reference)
Applies: original copy, 3x shift-scale-rotate, horizontal flip, vertical flip.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


def read_yolo_labels(txt_path: Path):
    boxes, classes = [], []
    if not txt_path.exists():
        return boxes, classes
    with open(txt_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) != 5:
                continue
            cls, x, y, w, h = parts
            classes.append(int(cls))
            boxes.append([float(x), float(y), float(w), float(h)])
    return boxes, classes


def write_yolo_labels(txt_path: Path, boxes, classes):
    with open(txt_path, 'w') as f:
        for (b, c) in zip(boxes, classes):
            x, y, w, h = b
            x = min(max(x, 0.0), 1.0)
            y = min(max(y, 0.0), 1.0)
            w = min(max(w, 0.0), 1.0)
            h = min(max(h, 0.0), 1.0)
            if w < MIN_WH or h < MIN_WH:
                continue
            f.write(f'{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n')


ssr_transform = A.Compose(
    [
        A.ShiftScaleRotate(
            shift_limit=SHIFT_LIMIT,
            scale_limit=SCALE_LIMIT,
            rotate_limit=ROTATE_LIMIT,
            border_mode=cv2.BORDER_CONSTANT,
            value=0,
            p=1.0,
        )
    ],
    bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=MIN_VIS),
)
hflip = A.Compose(
    [A.HorizontalFlip(p=1.0)],
    bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=MIN_VIS),
)
vflip = A.Compose(
    [A.VerticalFlip(p=1.0)],
    bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=MIN_VIS),
)


def apply_aug(image_bgr, bboxes, classes, transform):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    out = transform(image=image_rgb, bboxes=bboxes, class_labels=classes)
    image_bgr_out = cv2.cvtColor(out['image'], cv2.COLOR_RGB2BGR)
    return image_bgr_out, out['bboxes'], out['class_labels']


def augment_dataset(raw_im_dir: Path, raw_lb_dir: Path, aug_im_dir: Path, aug_lb_dir: Path):
    images = sorted([x for x in raw_im_dir.glob('*') if x.suffix.lower() in ['.png', '.jpg', '.jpeg']])
    assert images, f'No raw images in {raw_im_dir}'

    # Reuse already-built augmentations when requested
    if REUSE_AUGMENTATIONS:
        aug_images_existing = sorted([x for x in aug_im_dir.glob('*') if x.suffix.lower() in ['.png', '.jpg', '.jpeg']]) if aug_im_dir.exists() else []
        if len(aug_images_existing) > 0:
            print(f'Reusing augmentations from {aug_im_dir} ({len(aug_images_existing)} images)')
            return len(images), len(aug_images_existing)

    # Otherwise rebuild augmentations from scratch
    reset_dir(aug_im_dir)
    reset_dir(aug_lb_dir)

    total_written = 0
    for im_path in tqdm(images, desc=f'Augmenting {raw_im_dir.parent.name}'):
        base = im_path.stem
        ext = im_path.suffix.lower()
        lb_path = raw_lb_dir / f'{base}.txt'

        img = cv2.imread(str(im_path), cv2.IMREAD_COLOR)
        if img is None:
            continue

        bboxes, classes = read_yolo_labels(lb_path)

        cv2.imwrite(str(aug_im_dir / f'{base}{ext}'), img)
        write_yolo_labels(aug_lb_dir / f'{base}.txt', bboxes, classes)
        total_written += 1

        for i in range(N_SSR):
            aug_img, aug_boxes, aug_cls = apply_aug(img, bboxes, classes, ssr_transform)
            cv2.imwrite(str(aug_im_dir / f'{base}_ssr{i+1}.jpg'), aug_img)
            write_yolo_labels(aug_lb_dir / f'{base}_ssr{i+1}.txt', aug_boxes, aug_cls)
            total_written += 1

        aug_img, aug_boxes, aug_cls = apply_aug(img, bboxes, classes, hflip)
        cv2.imwrite(str(aug_im_dir / f'{base}_hflip.jpg'), aug_img)
        write_yolo_labels(aug_lb_dir / f'{base}_hflip.txt', aug_boxes, aug_cls)
        total_written += 1

        aug_img, aug_boxes, aug_cls = apply_aug(img, bboxes, classes, vflip)
        cv2.imwrite(str(aug_im_dir / f'{base}_vflip.jpg'), aug_img)
        write_yolo_labels(aug_lb_dir / f'{base}_vflip.txt', aug_boxes, aug_cls)
        total_written += 1

    return len(images), total_written


def split_dataset(aug_im_dir: Path, aug_lb_dir: Path, dataset_root: Path):
    for split in ['train', 'val', 'test']:
        reset_dir(dataset_root / split / 'images')
        reset_dir(dataset_root / split / 'labels')

    images = sorted([x for x in aug_im_dir.glob('*') if x.suffix.lower() in ['.png', '.jpg', '.jpeg']])
    assert images, f'No augmented images in {aug_im_dir}'

    random.shuffle(images)
    n_total = len(images)
    n_train = int(n_total * TRAIN_RATIO)
    n_val = int(n_total * VAL_RATIO)

    split_map = {
        'train': images[:n_train],
        'val': images[n_train:n_train + n_val],
        'test': images[n_train + n_val:],
    }

    for split, files in split_map.items():
        for im_path in tqdm(files, desc=f'Copying {dataset_root.name}/{split}'):
            stem = im_path.stem
            lb_path = aug_lb_dir / f'{stem}.txt'

            out_im = dataset_root / split / 'images' / im_path.name
            out_lb = dataset_root / split / 'labels' / f'{stem}.txt'

            shutil.copy2(im_path, out_im)
            if lb_path.exists():
                shutil.copy2(lb_path, out_lb)
            else:
                out_lb.write_text('')

    return {k: len(v) for k, v in split_map.items()}


def write_yaml(dataset_root: Path):
    yaml_path = dataset_root / 'data.yaml'
    yaml_text = f'''# Auto-generated for yolo12\npath: .\ntrain: {(dataset_root / 'train' / 'images').as_posix()}\nval: {(dataset_root / 'val' / 'images').as_posix()}\ntest: {(dataset_root / 'test' / 'images').as_posix()}\n\nnames:\n  0: class_0\n'''
    yaml_path.write_text(yaml_text)
    return yaml_path


assert WEIGHTS.exists(), f'Weights not found: {WEIGHTS}'

run_summaries = []
curve_frames = {}

for model_name, model_views in MODEL_SPECS.items():
    print('\n' + '=' * 80)
    print(f'Model: {model_name} | views={model_views}')

    model_root = DATA_ROOT / model_name
    raw_im_dir = model_root / 'raw' / 'images'
    raw_lb_dir = model_root / 'raw' / 'labels'
    aug_im_dir = model_root / 'aug' / 'images'
    aug_lb_dir = model_root / 'aug' / 'labels'

    copied, skipped = collect_pairs_for_views(model_views, raw_im_dir, raw_lb_dir)
    print(f'Collected pairs: {copied}, skipped without label: {skipped}')

    n_raw, n_aug = augment_dataset(raw_im_dir, raw_lb_dir, aug_im_dir, aug_lb_dir)
    print(f'Raw images: {n_raw}, after augmentation: {n_aug}')

    split_sizes = split_dataset(aug_im_dir, aug_lb_dir, model_root)
    print('Split sizes:', split_sizes)

    yaml_path = write_yaml(model_root)
    print('YAML:', yaml_path)

    model = YOLO(str(WEIGHTS))
    results = model.train(
        data=str(yaml_path),
        imgsz=IMGSZ,
        epochs=EPOCHS,
        batch=BATCH,
        workers=WORKERS,
        project=str(RUNS_ROOT),
        name=f'exp_{model_name}',
        pretrained=True,
        patience=20,
    )

    run_dir = Path(results.save_dir)
    best = run_dir / 'weights' / 'best.pt'
    assert best.exists(), f'best.pt missing for {model_name}'

    best_model = YOLO(str(best))
    val_metrics = best_model.val(data=str(yaml_path), split='val', imgsz=IMGSZ)
    test_metrics = best_model.val(data=str(yaml_path), split='test', imgsz=IMGSZ)

    # test loss as sum of loss components
    test_loss = (
        float(test_metrics.box_loss)
        + float(test_metrics.cls_loss)
        + float(test_metrics.dfl_loss)
    )

    results_csv = run_dir / 'results.csv'
    assert results_csv.exists(), f'results.csv missing for {model_name}'
    df = pd.read_csv(results_csv)
    curve_frames[model_name] = df

    run_summaries.append({
        'model': model_name,
        'views': ','.join(model_views),
        'run_dir': str(run_dir),
        'train_loss_final': float(df['train/box_loss'].iloc[-1] + df['train/cls_loss'].iloc[-1] + df['train/dfl_loss'].iloc[-1]),
        'val_loss_final': float(df['val/box_loss'].iloc[-1] + df['val/cls_loss'].iloc[-1] + df['val/dfl_loss'].iloc[-1]),
        'test_loss': test_loss,
    })

summary_df = pd.DataFrame(run_summaries)
summary_df

## 3) Training summary for all 4 models

In [ ]:
summary_df = summary_df.sort_values('model').reset_index(drop=True)
summary_df

## 4) Compare losses across axial/coronal/sagittal/all_views

In [ ]:
# Compare train/val curves across all models
train_cols = ['train/box_loss', 'train/cls_loss', 'train/dfl_loss']
val_cols = ['val/box_loss', 'val/cls_loss', 'val/dfl_loss']

plt.figure(figsize=(11, 5))
for model_name, df in curve_frames.items():
    train_total = df[train_cols].sum(axis=1)
    x = df['epoch'] if 'epoch' in df.columns else range(len(df))
    plt.plot(x, train_total, label=f'{model_name}')
plt.title('Train total loss comparison')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(11, 5))
for model_name, df in curve_frames.items():
    val_total = df[val_cols].sum(axis=1)
    x = df['epoch'] if 'epoch' in df.columns else range(len(df))
    plt.plot(x, val_total, label=f'{model_name}')
plt.title('Validation total loss comparison')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Test loss is measured once per model after training, so compare as bars
plt.figure(figsize=(8, 5))
plt.bar(summary_df['model'], summary_df['test_loss'])
plt.title('Test total loss comparison')
plt.xlabel('Model')
plt.ylabel('Loss')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5) Optional: run-level notes

In [ ]:
print('Run folders:')
for row in summary_df.itertuples(index=False):
    print(f"- {row.model}: {row.run_dir}")

## Optional detailed per-model curves

In [ ]:
# Optional detailed curves per model
for model_name, df in curve_frames.items():
    plt.figure(figsize=(10, 4))
    x = df['epoch'] if 'epoch' in df.columns else range(len(df))
    plt.plot(x, df['train/box_loss'], label='train/box_loss')
    plt.plot(x, df['val/box_loss'], label='val/box_loss')
    plt.title(f'{model_name}: box loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

## 6) Export summary table

In [ ]:
# Save summary CSV for quick comparison
summary_csv = RUNS_ROOT / 'selected_patients_4model_loss_summary.csv'
summary_df.to_csv(summary_csv, index=False)
print('Saved:', summary_csv)
summary_df

## Notes
- Uses only `p4, p5, p7, p12, p15, p16, p17` data.
- Trains 4 models: `axial`, `coronal`, `sagittal`, `all_views`.
- Weights are loaded from `pipeline_medsam2_3d/models/yolo12n/yolo12n.pt`.
- Augmentations match reference (`yolo_aug.py`): 3x SSR + hflip + vflip.
- Split matches reference (`splitting.py`): 85/7.5/7.5.
- Comparison plots include train loss, val loss (curves), and test loss (bars).